# 00 — Shared Embedding Foundation

**Purpose:** This notebook is the single source of truth for all embeddings.  
It loads the PriceRunner dataset, generates dense vector representations of every  
product title using `sentence-transformers`, saves them to disk, and runs a sanity check.  
Every downstream notebook (`01_classification`, `02_clustering`, `03_entity_matching`)  
and the Streamlit app loads those saved files — **we never re-embed**.

---

## Cell 1 — Library imports and version check

We check that the key libraries are importable before spending minutes on embedding.  
If any `ImportError` fires here, install the missing package and re-run this cell only.

In [ ]:
import importlib, sys

required = {
    'pandas'              : 'pandas',
    'numpy'               : 'numpy',
    'sentence_transformers': 'sentence-transformers',
    'sklearn'             : 'scikit-learn',
    'matplotlib'          : 'matplotlib',
}

missing = []
for module, pkg in required.items():
    try:
        importlib.import_module(module)
        print(f'  OK  {pkg}')
    except ImportError:
        print(f'  MISSING  {pkg}  →  pip install {pkg}')
        missing.append(pkg)

if missing:
    raise SystemExit(f'Install the missing packages then re-run: {missing}')
else:
    print('\nAll libraries present. Python', sys.version)

## Cell 2 — Load the dataset and inspect its structure

The CSV has leading whitespace in some column names (e.g., `' Category Label'`).  
We strip those immediately so downstream code can reference columns without  
worrying about invisible spaces.

In [ ]:
import pandas as pd
import numpy as np

CSV_PATH = 'pricerunner_aggregate.csv'

df = pd.read_csv(CSV_PATH)

# Strip leading/trailing whitespace from column names
df.columns = df.columns.str.strip()

print('=== Shape ===')
print(f'Rows: {df.shape[0]:,}   Columns: {df.shape[1]}')

print('\n=== Column names and dtypes ===')
print(df.dtypes)

print('\n=== First 5 rows ===')
df.head()

In [ ]:
print('=== Category distribution ===')
cat_counts = df['Category Label'].value_counts()
print(cat_counts.to_string())
print(f'\nTotal unique categories : {cat_counts.shape[0]}')

print('\n=== Merchant count ===')
print(f'Unique merchants: {df["Merchant ID"].nunique():,}')

print('\n=== Missing values ===')
print(df.isnull().sum())

## Cell 3 — Why Sentence Transformers instead of TF-IDF?

**TF-IDF** encodes documents as sparse vectors where each dimension is a vocabulary  
term weighted by how rare it is. Two titles that mean the same thing but use  
different words — *"iphone 14 pro max 256gb"* vs *"apple smartphone 14 pro 256 gb"* —  
share almost no overlapping tokens, so their TF-IDF cosine similarity collapses  
toward zero even though they refer to the same product.

**Sentence Transformers** (`all-MiniLM-L6-v2`) map text to 384-dimensional *dense*  
vectors that encode **meaning** rather than surface form. The model was pretrained  
on hundreds of millions of sentence pairs using contrastive learning, so semantically  
similar sentences land near each other in that space regardless of exact wording.

| Property | TF-IDF | `all-MiniLM-L6-v2` |
|---|---|---|
| Vector size | ~50 K (vocab) | 384 (fixed) |
| Sparsity | Sparse | Dense |
| Captures synonyms | ✗ | ✓ |
| Captures word order | ✗ | Partial |
| Needs GPU | No | No (fast on CPU) |
| Inference time (35K titles) | Instant | ~2–5 min on CPU |

For entity matching in particular, TF-IDF would fail badly because merchants  
often abbreviate brand names, reorder words, or use different model codes.  
Dense embeddings handle all of these gracefully.

## Cell 4 — Generate embeddings

**What happens here:**
1. We instantiate the `all-MiniLM-L6-v2` model. On first run it downloads ~90 MB  
   to `~/.cache/huggingface/`. Subsequent runs use the cache.
2. We call `.encode()` with `batch_size=256` and `show_progress_bar=True`.  
   Each batch of 256 titles is tokenised and forward-passed through the 6-layer  
   MiniLM transformer. On a modern CPU this takes roughly 2–5 minutes total.
3. The result is a `(35311, 384)` float32 NumPy array — one row per product title.

In [ ]:
from sentence_transformers import SentenceTransformer
import time

MODEL_NAME = 'all-MiniLM-L6-v2'

print(f'Loading model: {MODEL_NAME}')
model = SentenceTransformer(MODEL_NAME)

titles = df['Product Title'].tolist()
print(f'Encoding {len(titles):,} product titles …')

t0 = time.time()
embeddings = model.encode(
    titles,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=False,   # we'll normalise separately when needed
)
elapsed = time.time() - t0

print(f'\nDone in {elapsed:.1f}s')
print(f'Embeddings shape : {embeddings.shape}')   # expected (35311, 384)
print(f'dtype            : {embeddings.dtype}')   # expected float32
print(f'Memory footprint : {embeddings.nbytes / 1e6:.1f} MB')

## Cell 5 — Save embeddings and metadata to disk

**Why save?**  
Generating 35K embeddings takes 2–5 minutes every run. If each of the three  
pipeline notebooks re-ran the encoder, we'd waste 10–15 minutes on startup  
alone. Saving once and loading a `numpy` file (< 1 second) is the right trade-off.

**What we save:**
- `embeddings.npy` — the `(35311, 384)` float32 array
- `metadata.csv` — the four columns every other notebook needs:  
  `product_id`, `title`, `category`, `merchant`

In [ ]:
import os

# ── Save embeddings ──────────────────────────────────────────────────────────
EMBS_PATH = 'embeddings.npy'
np.save(EMBS_PATH, embeddings)
size_mb = os.path.getsize(EMBS_PATH) / 1e6
print(f'Saved embeddings → {EMBS_PATH}  ({size_mb:.1f} MB)')

# ── Save clean metadata ──────────────────────────────────────────────────────
META_PATH = 'metadata.csv'
meta = df[['Product ID', 'Product Title', 'Category Label', 'Merchant ID']].copy()
meta.columns = ['product_id', 'title', 'category', 'merchant_id']

# Attach merchant name if the column exists (it may be encoded in Merchant ID)
if 'Cluster Label' in df.columns:
    meta['cluster_label'] = df['Cluster Label'].values

meta.to_csv(META_PATH, index=False)
print(f'Saved metadata   → {META_PATH}')

print('\nMetadata columns :', list(meta.columns))
meta.head()

## Cell 6 — Reload check

Before the sanity check, let's verify we can actually load both files from a  
cold state (the way downstream notebooks will).

In [ ]:
# ── Simulate a cold load ─────────────────────────────────────────────────────
emb_loaded = np.load('embeddings.npy')
meta_loaded = pd.read_csv('metadata.csv')

assert emb_loaded.shape == embeddings.shape, 'Shape mismatch after reload!'
assert len(meta_loaded) == len(df),          'Row count mismatch after reload!'

print('Reload check passed ✓')
print(f'  embeddings : {emb_loaded.shape}')
print(f'  metadata   : {meta_loaded.shape}')

## Cell 7 — Sanity check: 5 nearest neighbours for 3 random titles

**Why this check matters:**  
If the embeddings were meaningless (e.g., random noise), the nearest neighbours  
of a laptop would be a pair of headphones or a washing machine.  
If the embeddings encode semantics correctly, the nearest neighbours of any title  
should be products from the same category (or at least conceptually related).  

**Method:** We use `sklearn`'s `cosine_similarity` on a sample — no FAISS needed  
for 3 queries against 35K vectors; it runs in under a second.

**How to interpret the output:**  
Look at the category column of the 5 neighbours. If they match (or are related to)  
the query's category, the embedding layer is working as expected.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

np.random.seed(42)
query_indices = np.random.choice(len(meta_loaded), size=3, replace=False)

print('=' * 70)
for qi in query_indices:
    query_vec  = emb_loaded[qi:qi+1]          # shape (1, 384)
    sims       = cosine_similarity(query_vec, emb_loaded)[0]   # (35311,)
    sims[qi]   = -1                            # exclude self
    top5_idx   = np.argsort(sims)[::-1][:5]

    q_row = meta_loaded.iloc[qi]
    print(f"\nQUERY  [{qi:>5}]  category: {q_row['category']:<25}  "
          f"title: {q_row['title']}")
    print('-' * 70)
    for rank, ni in enumerate(top5_idx, 1):
        n_row = meta_loaded.iloc[ni]
        print(f"  #{rank}  sim={sims[ni]:.4f}  [{n_row['category']:<25}]  "
              f"{n_row['title']}")
    print('=' * 70)

## Cell 8 — Visualise category distribution (quick bar chart)

Not required by the pipeline, but useful to see class balance before moving  
to classification. Imbalanced categories affect classifier performance and  
motivate stratified splitting in Notebook 01.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, ax = plt.subplots(figsize=(10, 4))

cat_counts_sorted = cat_counts.sort_values(ascending=True)
bars = ax.barh(
    cat_counts_sorted.index,
    cat_counts_sorted.values,
    color='steelblue',
    edgecolor='white',
    linewidth=0.5,
)

# Annotate each bar with its count
for bar, val in zip(bars, cat_counts_sorted.values):
    ax.text(
        bar.get_width() + 30, bar.get_y() + bar.get_height() / 2,
        f'{val:,}', va='center', ha='left', fontsize=9
    )

ax.set_xlabel('Number of product offers')
ax.set_title('PriceRunner dataset — offers per category', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('category_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved → category_distribution.png')

## Summary

| File created | Contents | Used by |
|---|---|---|
| `embeddings.npy` | `(35311, 384)` float32 array | All three pipelines + Streamlit |
| `metadata.csv` | `product_id`, `title`, `category`, `merchant_id` | All three pipelines + Streamlit |
| `category_distribution.png` | Bar chart of class counts | Reference only |

**Next step →** Open `01_classification.ipynb` to train a Logistic Regression  
classifier on top of these embeddings.